# Preprocessing 

What this script does:
  1. Walks data/raw/images/ and maps every sub-category to its 6 main classes
  2. Splits file paths into train / val / test using stratified sampling
     so every class keeps the exact 70 / 15 / 15 ratio
  3. Writes data/splits.csv with columns:
       filepath | main_class | class_idx | split
 
What this script does NOT do:
  - No image resizing or pixel changes  →  DataLoader handles that at training time
  - No copying of files                 →  filepaths still point to data/raw/
  - No augmentation                     →  each member's train.py handles that

After running:
  git add data/splits.csv
  git commit -m "add splits.csv (seed=42, 70/15/15)"
  git push
  → all members git pull and use the same split
 
# DO NOT re-run this again - one time only by one member 

In [ ]:
import yaml
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
 

In [ ]:
import os
os.chdir("..")

In [ ]:
# Guard code only
from pathlib import Path

splits_csv = Path("data/splits.csv")

if splits_csv.exists():
    print("splits.csv already exists — no need to re-run.")
    print("Just use the existing file. Only continue if you intentionally want to reset splits.")
    raise SystemExit("Stopping to protect existing splits.")

In [ ]:
#Load shared config 
with open("config.yaml") as f:
    cfg = yaml.safe_load(f)
 
RAW_DIR    = Path(cfg["data"]["raw_dir"]) / "images"   # data/raw/images/
SPLITS_CSV = Path(cfg["data"]["splits_csv"])            # data/splits.csv
SEED       = cfg["data"]["seed"]                        # 42 — NEVER change
TRAIN_R    = cfg["data"]["train_ratio"]                 # 0.70
VAL_R      = cfg["data"]["val_ratio"]                   # 0.15
TEST_R     = cfg["data"]["test_ratio"]                  # 0.15
CLASS_MAP  = cfg["classes"]                             # {"Plastic": 0, ...}

In [ ]:
# basic sanity check
assert abs(TRAIN_R + VAL_R + TEST_R - 1.0) < 1e-6, \
    "train_ratio + val_ratio + test_ratio must sum to 1.0 in config.yaml"

In [ ]:
# Keyword → main class mapping 
KEYWORD_TO_CLASS = {
    # Plastic (9 sub-categories)
    "plastic"      : "Plastic",
    "bag"          : "Plastic",
    "straw"        : "Plastic",
    "cup lid"      : "Plastic",
    "cutlery"      : "Plastic",
    "container"    : "Plastic",
    # Paper (5 sub-categories)
    "newspaper"    : "Paper",
    "office paper" : "Paper",
    "magazine"     : "Paper",
    "cardboard"    : "Paper",
    # Glass (3 sub-categories)
    "glass"        : "Glass",
    # Metal (4 sub-categories)
    "aluminum"     : "Metal",
    "steel"        : "Metal",
    "aerosol"      : "Metal",
    "metal"        : "Metal",
    # Organic (5 sub-categories)
    "food waste"   : "Organic",
    "eggshell"     : "Organic",
    "coffee"       : "Organic",
    "tea bag"      : "Organic",
    "fruit"        : "Organic",
    "vegetable"    : "Organic",
    # Textile (2 sub-categories)
    "clothing"     : "Textile",
    "shoe"         : "Textile",
}
 
def get_main_class(subfolder_name: str) -> str:
    """Return the main class label for a sub-category folder name.
    Returns 'Unknown' if no keyword matches — script will stop with an error.
    """
    name_lower = subfolder_name.lower()
    for keyword, main_class in KEYWORD_TO_CLASS.items():
        if keyword in name_lower:
            return main_class
    return "Unknown"

In [ ]:
#Step 1: Collect all image records from raw directory
print("=" * 58)
print("  prepare_data.py  —  Split & CSV generation")
print("=" * 58)
print(f"\n[1/4] Scanning: {RAW_DIR}\n")
 
if not RAW_DIR.exists():
    raise FileNotFoundError(
        f"\nDirectory not found: {RAW_DIR}\n"
        "Make sure the Kaggle dataset is unzipped so that:\n"
        "  data/raw/images/<sub-category>/default/*.png  exists."
    )
 
records = []
for subfolder in sorted(RAW_DIR.iterdir()):
    if not subfolder.is_dir():
        continue
 
    main_class = get_main_class(subfolder.name)
 
    # each sub-category has two sub-folders: default and real_world
    for image_type in ["default", "real_world"]:
        type_dir = subfolder / image_type
        if not type_dir.exists():
            continue
        for img_path in sorted(type_dir.glob("*.png")):
            records.append({
                "filepath"   : str(img_path),       # relative path from project root
                "main_class" : main_class,           # one of 6 class names
                "class_idx"  : CLASS_MAP.get(main_class, -1),  # integer label 0-5
                # subfolder and image_type kept for debugging — not used in training
                "subfolder"  : subfolder.name,
                "image_type" : image_type,
            })
 
df_all = pd.DataFrame(records)
print(f"  Total images collected : {len(df_all)}")
print(f"  Sub-categories found   : {df_all['subfolder'].nunique()}")

In [ ]:
#Guard: check for unmapped sub-categories
unknown =Guard: check for unmapped sub-categories df_all[df_all["main_class"] == "Unknown"]
if len(unknown) > 0:
    print(f"\n  ERROR — {len(unknown)} images not mapped to any class:")
    for name in unknown["subfolder"].unique():
        print(f"    '{name}'")
    print("\n  Add the missing keyword to KEYWORD_TO_CLASS and re-run.")
    raise SystemExit(1)

In [ ]:
#Guard: check all class_idx values are valid (0–5)
invalid = df_all[df_all["class_idx"] == -1]
if len(invalid) > 0:
    print(f"\n  ERROR — class_idx = -1 for these classes (not in config.yaml):")
    print(invalid["main_class"].unique())
    raise SystemExit(1)
 
print("  All sub-categories mapped to 6 main classes. No unknowns.\n")

In [ ]:
# Quick per-class count before splitting
print("  Images per class (pre-split):")
for cls, count in df_all["main_class"].value_counts().reindex(CLASS_MAP.keys()).items():
    print(f"    {cls:<10}  {count}")

In [ ]:
# ── Step 2: Stratified split — train / val / test ──────────────────────────────
# Stratified on class_idx so every class gets the same 70/15/15 proportion.
#
# We do it in two stages because train_test_split only splits into two parts:
#   Stage A:  full (100%) → train_val (85%)  +  test (15%)
#   Stage B:  train_val   → train (70%)      +  val  (15%)
#
# The val size in Stage B is computed relative to train_val:
#   val_relative = 0.15 / (0.70 + 0.15)  =  0.15 / 0.85  ≈  0.1765
 
print(f"\n[2/4] Stratified split  (seed={SEED}) ...")
 
# Stage A — carve out test set
df_train_val, df_test = train_test_split(
    df_all,
    test_size=TEST_R,                   # 15% goes to test
    random_state=SEED,
    stratify=df_all["class_idx"],       # preserve class balance
)
 
# Stage B — split remainder into train and val
val_relative = VAL_R / (TRAIN_R + VAL_R)   # ≈ 0.1765
df_train, df_val = train_test_split(
    df_train_val,
    test_size=val_relative,
    random_state=SEED,
    stratify=df_train_val["class_idx"],
)
 
# Tag each subset with its split name
df_train = df_train.copy(); df_train["split"] = "train"
df_val   = df_val.copy();   df_val["split"]   = "val"
df_test  = df_test.copy();  df_test["split"]  = "test"
 
# Merge back into one DataFrame and shuffle rows
df_splits = (
    pd.concat([df_train, df_val, df_test], ignore_index=True)
    .sample(frac=1, random_state=SEED)
    .reset_index(drop=True)
)

In [ ]:
#Step 3: Verify split sizes and class balance
print("\n[3/4] Verifying split ...\n")
 
print("  Split sizes:")
split_order  = ["train", "val", "test"]
split_counts = df_splits["split"].value_counts().reindex(split_order)
for name, count in split_counts.items():
    pct = 100 * count / len(df_splits)
    print(f"    {name:<6}  {count:>5} images  ({pct:.1f}%)")
print(f"    {'TOTAL':<6}  {len(df_splits):>5} images")
 
print("\n  Class balance per split  (should be roughly equal across rows):")
balance = (
    df_splits
    .groupby(["split", "main_class"])
    .size()
    .unstack(fill_value=0)
    .reindex(split_order)            # train / val / test order
    [list(CLASS_MAP.keys())]         # column order matches config
)
print(balance.to_string())
 
# Warn if any class count deviates more than 20% from the expected mean
print()
all_ok = True
for split_name in split_order:
    sub = df_splits[df_splits["split"] == split_name]
    counts     = sub["main_class"].value_counts()
    mean_count = counts.mean()
    for cls, cnt in counts.items():
        deviation = abs(cnt - mean_count) / mean_count
        if deviation > 0.20:
            print(
                f"  WARNING: {cls} in '{split_name}' has {cnt} images "
                f"({deviation*100:.0f}% off the mean). "
                "Check KEYWORD_TO_CLASS for missing sub-categories."
            )
            all_ok = False
 
if all_ok:
    print("  Class balance looks good — no large deviations detected.")

In [ ]:
#Step 4: Save splits.csv
print(f"\n[4/4] Saving splits.csv → {SPLITS_CSV}")

In [ ]:
# Only write the columns that training scripts actually need
output_cols = ["filepath", "main_class", "class_idx", "split"]
df_out = df_splits[output_cols]
 
SPLITS_CSV.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(SPLITS_CSV, index=False)
 
print(f"  Rows    : {len(df_out)}")
print(f"  Columns : {list(df_out.columns)}")
print(f"\n  Preview (first 6 rows):")
print(df_out.head(6).to_string(index=False))